# ARIA Phase 4 - Prophet Demand Proxy v1

This notebook creates the Phase 4 demand module for the ARIA KPMG Capstone project. It trains exactly two city-level Prophet models: one for Paris and one for Athens. Neighbourhood rows are produced by scaling each city-level Prophet shape using neighbourhood annual occupancy baselines and capped review-growth momentum.

## Methodological Honesty Statement

The available source dataset is cross-sectional: one row per listing. It has no true `ds` time column, no booking calendar, no review-date history, and no repeated monthly observations. Therefore this notebook does **not** train a true historical booking/calendar forecast. It constructs a scenario-based Prophet demand proxy from annual occupancy estimates, assumed STR seasonality, and capped review-growth momentum. It is not trained on observed booking/calendar time series. Forecast validation in this notebook is an internal consistency check only, not real out-of-sample market validation.

## Assumptions Table

| Assumption | Implementation | Rationale |
|---|---|---|
| Synthetic monthly range | Jan 2021 to Sep 2025 for training; Oct 2025 to Sep 2026 for 12-month forecast | Gives Prophet a monthly series while making clear it is constructed, not observed |
| Seasonality assumptions | City-specific monthly STR multipliers, normalized to annual mean 1.0 | Reflects known tourist seasonality patterns for Paris and Athens without claiming learned booking seasonality |
| Momentum treatment | City/neighbourhood median `review_growth_24_25`, capped by city quantiles and hard limits, converted to a modest multiplier with slope 0.015 and bounds 0.90-1.12 | Uses review growth as directional demand momentum, not literal demand growth |
| Occupancy cap | Forecasts clipped to 70% of days in each month | Aligns with the 255-day annual cap visible in occupancy estimates |
| City-level Prophet only | Exactly one Prophet model per city; neighbourhood rows are scaled from city shape | Avoids false precision from fitting neighbourhood models to synthetic series |
| Validation interpretation | Sanity checks only: schema, row count, cap compliance, interval order, null checks | No observed monthly history exists, so real forecast accuracy cannot be measured |

## 1. Imports, Paths, and Constants

In [ ]:
from pathlib import Path
import pickle
import warnings

import numpy as np
import pandas as pd
from prophet import Prophet

warnings.filterwarnings('ignore', category=FutureWarning)

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        data_file = candidate / 'data' / 'processed' / 'aria_mega_dataset_v4_1_final.csv'
        if data_file.exists():
            return candidate
    raise FileNotFoundError('Could not locate repository root containing data/processed/aria_mega_dataset_v4_1_final.csv')


ROOT = find_repo_root(Path.cwd().resolve())
DATA_PATH = ROOT / 'data' / 'processed' / 'aria_mega_dataset_v4_1_final.csv'
MODEL_PATHS = {
    'paris': ROOT / 'models' / 'prophet_paris_v1.pkl',
    'athens': ROOT / 'models' / 'prophet_athens_v1.pkl',
}
OUTPUT_PATHS = {
    'paris': ROOT / 'data' / 'outputs' / 'prophet_paris_forecast_v1.csv',
    'athens': ROOT / 'data' / 'outputs' / 'prophet_athens_forecast_v1.csv',
}

TRAINING_START = pd.Timestamp('2021-01-01')
TRAINING_END = pd.Timestamp('2025-09-01')
FORECAST_PERIODS = 12
MAX_OCCUPANCY_SHARE = 0.70
MOMENTUM_SLOPE = 0.015
MOMENTUM_MIN = 0.90
MOMENTUM_MAX = 1.12
METHODOLOGY_NOTE = (
    'Scenario-based Prophet demand proxy built from annual occupancy estimates, '
    'assumed STR seasonality, and capped review-growth momentum. Not trained on '
    'observed booking/calendar time series.'
)

SEASONALITY = {
    'paris': {
        1: 0.78, 2: 0.84, 3: 0.95, 4: 1.05, 5: 1.10, 6: 1.14,
        7: 1.08, 8: 0.92, 9: 1.12, 10: 1.06, 11: 0.94, 12: 1.02,
    },
    'athens': {
        1: 0.65, 2: 0.70, 3: 0.85, 4: 1.00, 5: 1.15, 6: 1.25,
        7: 1.35, 8: 1.30, 9: 1.15, 10: 1.00, 11: 0.80, 12: 0.80,
    },
}

OUTPUT_COLUMNS = [
    'ds', 'city', 'neighbourhood', 'forecast_month', 'model_level',
    'is_scenario_proxy', 'yhat', 'yhat_lower', 'yhat_upper',
    'monthly_baseline', 'annual_occupancy_baseline',
    'review_growth_24_25_raw', 'review_growth_24_25_capped',
    'seasonality_multiplier', 'momentum_multiplier', 'n_listings',
    'training_start', 'training_end', 'methodology_note',
]

## 2. Reusable Pipeline Functions

The functions below aggregate eligible listing rows, construct the deterministic monthly city-level scenario series, fit Prophet, scale city forecasts to neighbourhood rows, and enforce the required output schema.

In [ ]:
def momentum_multiplier(capped_growth):
    return float(np.clip(1.0 + MOMENTUM_SLOPE * capped_growth, MOMENTUM_MIN, MOMENTUM_MAX))


def build_city_artifacts(df, city):
    city_df = df[df['city'].str.lower() == city].copy()
    if city_df.empty:
        raise ValueError(f'No eligible rows found for {city}')

    city_growth = city_df['review_growth_24_25'].dropna()
    if city_growth.empty:
        lower_cap, upper_cap, city_growth_raw = -1.0, 10.0, 0.0
    else:
        lower_cap = max(float(city_growth.quantile(0.05)), -1.0)
        upper_cap = min(float(city_growth.quantile(0.95)), 10.0)
        if lower_cap >= upper_cap:
            lower_cap, upper_cap = -1.0, 10.0
        city_growth_raw = float(city_growth.median())

    city_growth_capped = float(np.clip(city_growth_raw, lower_cap, upper_cap))
    city_momentum = momentum_multiplier(city_growth_capped)
    city_annual_baseline = float(city_df['estimated_occupancy_l365d'].median())
    city_monthly_baseline = city_annual_baseline / 12.0

    neigh = (
        city_df.groupby('neighbourhood', dropna=False)
        .agg(
            n_listings=('listing_id', 'count'),
            annual_occupancy_baseline=('estimated_occupancy_l365d', 'median'),
            review_growth_24_25_raw=('review_growth_24_25', 'median'),
        )
        .reset_index()
    )
    neigh['neighbourhood'] = neigh['neighbourhood'].fillna('Unknown')
    neigh['review_growth_24_25_raw'] = neigh['review_growth_24_25_raw'].fillna(city_growth_raw)
    neigh['review_growth_24_25_capped'] = neigh['review_growth_24_25_raw'].clip(lower_cap, upper_cap)
    neigh['momentum_multiplier'] = neigh['review_growth_24_25_capped'].apply(momentum_multiplier)
    neigh['monthly_baseline'] = neigh['annual_occupancy_baseline'] / 12.0

    months = pd.date_range(TRAINING_START, TRAINING_END, freq='MS')
    training = pd.DataFrame({'ds': months})
    training['seasonality_multiplier'] = training['ds'].dt.month.map(SEASONALITY[city]).astype(float)
    progress = np.linspace(0.0, 1.0, len(training))
    training['momentum_multiplier'] = 1.0 + (city_momentum - 1.0) * progress
    training['y'] = city_monthly_baseline * training['seasonality_multiplier'] * training['momentum_multiplier']
    training['cap'] = training['ds'].dt.days_in_month * MAX_OCCUPANCY_SHARE
    training['floor'] = 0.0
    training['y'] = training[['y', 'cap']].min(axis=1)

    model = Prophet(
        growth='linear',
        yearly_seasonality=5,
        weekly_seasonality=False,
        daily_seasonality=False,
        seasonality_mode='multiplicative',
        interval_width=0.80,
        changepoint_prior_scale=0.02,
        seasonality_prior_scale=5.0,
    )
    model.fit(training[['ds', 'y']])

    future = model.make_future_dataframe(periods=FORECAST_PERIODS, freq='MS', include_history=False)
    city_forecast = model.predict(future)[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].copy()
    city_forecast['seasonality_multiplier'] = city_forecast['ds'].dt.month.map(SEASONALITY[city]).astype(float)

    rows = []
    for _, nrow in neigh.iterrows():
        baseline_scale = float(nrow['monthly_baseline']) / city_monthly_baseline if city_monthly_baseline > 0 else 1.0
        momentum_scale = float(nrow['momentum_multiplier']) / city_momentum if city_momentum > 0 else 1.0
        for _, frow in city_forecast.iterrows():
            cap = float(frow['ds'].days_in_month * MAX_OCCUPANCY_SHARE)
            yhat = float(np.clip(frow['yhat'] * baseline_scale * momentum_scale, 0.0, cap))
            yhat_lower = float(np.clip(frow['yhat_lower'] * baseline_scale * momentum_scale, 0.0, cap))
            yhat_upper = float(np.clip(frow['yhat_upper'] * baseline_scale * momentum_scale, 0.0, cap))
            yhat_lower = min(yhat_lower, yhat)
            yhat_upper = max(yhat_upper, yhat)
            rows.append({
                'ds': frow['ds'].strftime('%Y-%m-%d'),
                'city': city,
                'neighbourhood': nrow['neighbourhood'],
                'forecast_month': frow['ds'].strftime('%Y-%m'),
                'model_level': 'city_prophet_scaled_neighbourhood',
                'is_scenario_proxy': True,
                'yhat': round(yhat, 4),
                'yhat_lower': round(yhat_lower, 4),
                'yhat_upper': round(yhat_upper, 4),
                'monthly_baseline': round(float(nrow['monthly_baseline']), 4),
                'annual_occupancy_baseline': round(float(nrow['annual_occupancy_baseline']), 4),
                'review_growth_24_25_raw': round(float(nrow['review_growth_24_25_raw']), 4),
                'review_growth_24_25_capped': round(float(nrow['review_growth_24_25_capped']), 4),
                'seasonality_multiplier': round(float(frow['seasonality_multiplier']), 4),
                'momentum_multiplier': round(float(nrow['momentum_multiplier']), 4),
                'n_listings': int(nrow['n_listings']),
                'training_start': TRAINING_START.strftime('%Y-%m-%d'),
                'training_end': TRAINING_END.strftime('%Y-%m-%d'),
                'methodology_note': METHODOLOGY_NOTE,
            })

    output = pd.DataFrame(rows, columns=OUTPUT_COLUMNS)
    bundle = {
        'city': city,
        'model_level': 'city',
        'model': model,
        'training_series': training,
        'city_forecast': city_forecast,
        'neighbourhood_baselines': neigh,
        'assumptions': {
            'methodology_note': METHODOLOGY_NOTE,
            'training_start': TRAINING_START.strftime('%Y-%m-%d'),
            'training_end': TRAINING_END.strftime('%Y-%m-%d'),
            'forecast_periods': FORECAST_PERIODS,
            'seasonality': SEASONALITY[city],
            'momentum_slope': MOMENTUM_SLOPE,
            'momentum_min': MOMENTUM_MIN,
            'momentum_max': MOMENTUM_MAX,
            'review_growth_cap_lower': lower_cap,
            'review_growth_cap_upper': upper_cap,
            'max_occupancy_share_of_days_in_month': MAX_OCCUPANCY_SHARE,
            'no_random_noise': True,
            'true_time_axis_available': False,
            'observed_booking_calendar_available': False,
            'observed_review_dates_available': False,
            'validation_is_internal_consistency_only': True,
        },
        'city_baseline': {
            'annual_occupancy_baseline': city_annual_baseline,
            'monthly_baseline': city_monthly_baseline,
            'review_growth_24_25_raw': city_growth_raw,
            'review_growth_24_25_capped': city_growth_capped,
            'momentum_multiplier': city_momentum,
            'n_eligible_rows': int(len(city_df)),
            'n_neighbourhoods': int(neigh['neighbourhood'].nunique()),
        },
    }
    return output, bundle


def sanity_check(output, city):
    assert list(output.columns) == OUTPUT_COLUMNS, f'{city}: output columns do not match required schema'
    assert not output[OUTPUT_COLUMNS].isna().any().any(), f'{city}: required output columns contain nulls'
    assert (output.groupby('neighbourhood')['ds'].nunique() == FORECAST_PERIODS).all(), f'{city}: missing monthly rows'
    ds = pd.to_datetime(output['ds'])
    caps = ds.dt.days_in_month * MAX_OCCUPANCY_SHARE
    for col in ['yhat', 'yhat_lower', 'yhat_upper']:
        assert (output[col].to_numpy() <= caps.to_numpy() + 1e-9).all(), f'{city}: {col} exceeds 70 percent monthly cap'
    assert ((output['yhat_lower'] <= output['yhat']) & (output['yhat'] <= output['yhat_upper'])).all(), f'{city}: invalid interval order'
    assert output['is_scenario_proxy'].eq(True).all(), f'{city}: scenario proxy flag missing'
    assert output['methodology_note'].eq(METHODOLOGY_NOTE).all(), f'{city}: methodology note mismatch'

## 3. Load, Inspect, Train, Forecast, and Save

This cell writes only the five Phase 4 deliverables: the two model pickle files, the two forecast CSVs, and this notebook.

In [ ]:
df = pd.read_csv(DATA_PATH, low_memory=False)
print('Source shape:', df.shape)
print('Has true ds column:', 'ds' in df.columns)
print('Has last_review column:', 'last_review' in df.columns)

eligible = df[df['prophet_training_eligible'] == 1].copy()
eligibility_summary = eligible.groupby('city').agg(
    eligible_rows=('listing_id', 'count'),
    neighbourhoods=('neighbourhood', 'nunique'),
    median_annual_occupancy=('estimated_occupancy_l365d', 'median'),
    median_review_growth=('review_growth_24_25', 'median'),
)
print(eligibility_summary)

outputs = {}
bundles = {}
for city in ['paris', 'athens']:
    output, bundle = build_city_artifacts(eligible, city)
    sanity_check(output, city)
    output.to_csv(OUTPUT_PATHS[city], index=False)
    with open(MODEL_PATHS[city], 'wb') as f:
        pickle.dump(bundle, f)
    outputs[city] = output
    bundles[city] = bundle
    print(city, output.shape, MODEL_PATHS[city].name, OUTPUT_PATHS[city].name)

## 4. Business Interpretation for ARIA Agents

- **Investor agent:** Use these outputs as directional demand pressure by neighbourhood. A high `yhat` with strong `momentum_multiplier` suggests a demand-resilient neighbourhood, subject to regulatory and pricing checks.
- **Host agent:** Use the forecast as a pricing-pressure context layer. It should modify, not replace, XGBoost fair-price and LightGBM risk signals.
- **Developer agent:** Use neighbourhood-level scaled demand proxies to compare demand resilience across zones, while remembering that this is not observed calendar demand.

Do not present these outputs as real market forecast accuracy. They are scenario inputs for the multi-agent demo until booking calendar or review-date time series become available.

In [ ]:
for city, output in outputs.items():
    ranked = (
        output.groupby('neighbourhood')
        .agg(
            avg_monthly_yhat=('yhat', 'mean'),
            annual_occupancy_baseline=('annual_occupancy_baseline', 'first'),
            momentum_multiplier=('momentum_multiplier', 'first'),
            n_listings=('n_listings', 'first'),
        )
        .sort_values(['avg_monthly_yhat', 'momentum_multiplier'], ascending=False)
        .head(10)
    )
    print('\n', city.upper(), 'top scenario demand proxy neighbourhoods')
    print(ranked.round(3))

## 5. Final Caveat

The best future improvement is to replace the synthetic monthly series with actual monthly observations from Inside Airbnb `calendar.csv` or review-level data containing `listing_id` and `date`. Once a true `ds, y` table exists, Prophet can be retrained as a real historical forecast rather than a scenario proxy.